In [2]:
import pandas as pd
import numpy as np
import os
import torch
import warnings
import random
import joblib

from sentence_transformers import SentenceTransformer
from sklearn.metrics.pairwise import cosine_distances, cosine_similarity
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score
from sklearn.preprocessing import normalize
from sklearn.metrics import pairwise_distances_argmin

import matplotlib.pyplot as plt
import umap
import nltk
import torch
import torch.nn.functional as F
import hdbscan
from transformers import AutoTokenizer

#nltk.download('punkt')
#nltk.download('punkt_tab')



"""   

Copyright (c) 2026, Michael Tchuindjang
All rights reserved.

This code was developed as part of a PhD research project in Cybersecurity and Artificial Intelligence, 
supported by a studentship at the University of the West of England (UWE Bristol).

Use of this software is permitted for academic, educational, and research purposes.  
For any commercial use or redistribution, please contact the author for permission.

Disclaimer:
In no event shall the author or UWE be liable for any claim, damages, or other liability arising from the use of this code.

Acknowledgment of the author and the research context is appreciated in any derivative work or publication.


"""

# =========================
# GLOBAL PARAMETERS
# =========================
TRAIN_DIR = "training"
os.makedirs(TRAIN_DIR, exist_ok=True)

TESTS = ['Test_1', 
         'Test_2', 
         'Test_3', 
         'Test_4']

TRAINING_TEST = TESTS[1] # TESTS[1]: Test_2
# Tests/training files are formatted like Test_X_all_models.csv
TRAINING_TEST_FILE = TRAINING_TEST + '_all_models.csv'
INPUT_FILE = os.path.join(TRAIN_DIR, TRAINING_TEST_FILE)

EMB_MODEL_NAMES = [
    'all-MiniLM-L6-v2',
    'all-mpnet-base-v2',
    'all-roberta-large-v1'
]

EMBEDDING_MODEL_NAME = EMB_MODEL_NAMES[0]

embedder = SentenceTransformer(EMBEDDING_MODEL_NAME)
tokenizer = AutoTokenizer.from_pretrained("sentence-transformers/" + EMBEDDING_MODEL_NAME)

CHUNKING_POOLING = "norm_mean" # change: mean / weighted / max / norm_mean
CHUNKING_OVERLAP = 20  # 0, 20, 40, 60

USE_CHUNKING = False

CHUNK_TAG = "chunked" if USE_CHUNKING else "no_chunk"

USE_FULL_CONVERSATION = False

RUN_SIGNATURE = f"{EMBEDDING_MODEL_NAME}__{CHUNK_TAG}__full-{USE_FULL_CONVERSATION}__{TRAINING_TEST}"

EMB_MODEL_FOLDER = os.path.join(TRAIN_DIR, EMBEDDING_MODEL_NAME)
os.makedirs(EMB_MODEL_FOLDER, exist_ok=True)

OUTPUT_PREFIX = os.path.join(
    EMB_MODEL_FOLDER,
    f"{RUN_SIGNATURE}__model_agnostic_"
)


# =========================
# TEXT EXTRACTION
# =========================
def get_final_response(row):
    return row.get(f"output_turn_{row['turn_depth']}", "")

def get_full_conversation(row):
    conversation = []
    for i in range(1, row["turn_depth"] + 1):
        user_text = row.get(f"turn_{i}", "")
        assistant_text = row.get(f"output_turn_{i}", "")

        if pd.notna(user_text) and user_text.strip():
            conversation.append(f"USER: {user_text}")

        if pd.notna(assistant_text) and assistant_text.strip():
            conversation.append(f"ASSISTANT: {assistant_text}")

    return "\n".join(conversation)

def set_seed(seed=42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)

    # makes some ops deterministic (optional but good for research)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

# =========================
# CHUNKING ENCODER
# =========================
def embed_no_chunk(texts, model, tokenizer):
    """
    Encode list of texts using standard truncated embedding.
    
    Args:
        texts (list[str])
        model (SentenceTransformer)

    Returns:
        np.ndarray: shape (N, D)
    """
    return model.encode(texts, convert_to_numpy=True, batch_size=64, show_progress_bar=True)


def chunk_text_token_level(text, tokenizer, max_tokens=256, overlap=CHUNKING_OVERLAP):
    """
    Token-consistent chunking using model tokenizer.
    """
    tokens = tokenizer.encode(text, add_special_tokens=False)

    chunks = []
    start = 0

    while start < len(tokens):
        end = min(start + max_tokens, len(tokens))
        chunk_tokens = tokens[start:end]

        chunk = tokenizer.decode(chunk_tokens, skip_special_tokens=True)
        chunks.append(chunk.strip())

        start += max_tokens - overlap

    return chunks

# =========================================================
# 3. POOLING STRATEGIES (UNBIASED OPTIONS)
# =========================================================

def mean_pool(embeddings):
    return np.mean(embeddings, axis=0)

def weighted_mean_pool(embeddings, chunks, tokenizer):
    weights = np.array([
        len(tokenizer.encode(c, add_special_tokens=False))
        for c in chunks
    ])
    return np.average(embeddings, axis=0, weights=weights)

def max_pool(embeddings):
    return np.max(embeddings, axis=0)

def normalized_mean_pool(embeddings):
    emb = normalize(embeddings, axis=1)
    pooled = np.mean(emb, axis=0)
    return normalize(pooled.reshape(1, -1))[0]

# POOLING_METHOD is "weighted" by default  # change: mean / weighted / max / norm_mean
def embed_chunk(texts, model, tokenizer, pooling=CHUNKING_POOLING, batch_size=64):
    all_embeddings = []
    chunk_counts = []
    all_chunks = []

    # -----------------------------
    # Flatten chunks
    # -----------------------------
    for text in texts:
        chunks = chunk_text_token_level(text, tokenizer)

        if len(chunks) == 0:
            chunks = [""]

        chunk_counts.append(len(chunks))
        all_chunks.extend(chunks)

    # -----------------------------
    # Encode chunks
    # -----------------------------
    chunk_embeddings = model.encode(
        all_chunks,
        batch_size=batch_size,
        convert_to_numpy=True,
        show_progress_bar=True
    )

    # -----------------------------
    # Pool per document
    # -----------------------------
    idx = 0

    for i, count in enumerate(chunk_counts):
        doc_chunks = chunk_embeddings[idx:idx + count]
        idx += count

        if pooling == "mean":
            doc_emb = mean_pool(doc_chunks)

        elif pooling == "weighted":
            doc_emb = weighted_mean_pool(doc_chunks, all_chunks[:count], tokenizer)

        elif pooling == "max":
            doc_emb = max_pool(doc_chunks)

        elif pooling == "norm_mean":
            doc_emb = normalized_mean_pool(doc_chunks)

        else:
            raise ValueError("Unknown pooling method")

        all_embeddings.append(doc_emb)

    return np.array(all_embeddings)


# =========================
# ENCODING WRAPPER
# =========================
def encode_texts(texts, model, tokenizer):
    if USE_CHUNKING:
        embeddings = embed_chunk(texts, model, tokenizer)
        if isinstance(embeddings, torch.Tensor):
            embeddings = embeddings.detach().cpu().numpy()
        return embeddings
    return embed_no_chunk(texts, model, tokenizer)


# =========================
# CLUSTER CENTROIDS (NEW)
# =========================
def compute_cluster_centroids(X_strata, cluster_labels):
    """
    Compute centroids of refusal clusters (semantic refusal modes).

    Returns:
        np.ndarray of shape (num_clusters, dim)
    """
    centroids = []

    for c in np.unique(cluster_labels):
        if c == -1:
            continue  # skip noise

        cluster_points = X_strata[cluster_labels == c]

        if len(cluster_points) == 0:
            continue

        centroid = np.mean(cluster_points, axis=0)
        centroids.append(centroid)

    return np.vstack(centroids)
    
# =========================
# STRATA BUILDING (UPDATED: optional normalization)
# =========================
def build_strata_embeddings(df, embeddings, normalize_emb=True):

    df = df.copy()
    df["stratum_id"] = list(zip(df["subtopic"], df["tense"], df["turn_depth"]))

    if normalize_emb:
        embeddings = normalize(embeddings, axis=1)

    groups = df.groupby("stratum_id")

    X_strata = []
    strata_to_idx = []

    for _, group in groups:
        idx = group.index.values
        X_strata.append(embeddings[idx].mean(axis=0))
        strata_to_idx.append(idx)

    X_strata = np.array(X_strata)

    if normalize_emb:
        X_strata = normalize(X_strata, axis=1)

    return X_strata, strata_to_idx

# =========================
# CLUSTER STRATA (REFUSAL CLASSES)
# =========================
def cluster_strata(X_strata, method="kmeans", k=10):

    N = len(X_strata)

    if method == "kmeans":
        k = min(max(2, k), (N-1))
        model = KMeans(n_clusters=k, random_state=42, n_init=10)
        return model.fit_predict(X_strata)

    elif method == "hdbscan":
        # ---- k controls granularity (NOT cluster count) ----
        # bigger k → smaller clusters
        # smaller k → larger clusters
        min_cluster_size = max(5, int(N / (k * 2)))   # smooth scaling
        min_samples = max(2, int(k / 10))             # mild coupling

        model = hdbscan.HDBSCAN(
            min_cluster_size=min_cluster_size,
            min_samples=min_samples,
            metric="euclidean"
        )
        return model.fit_predict(X_strata)

    else:
        raise ValueError("method must be kmeans or hdbscan")

# =========================
# K-CENTER SELECTION (FIXED NORMALIZATION CONSISTENCY)
# =========================
def k_center_greedy(X, k):

    n = len(X)
    if n == 0:
        return []

    # ensure cosine consistency
    X = normalize(X, axis=1)

    selected = [np.random.randint(n)]

    min_dist = cosine_distances(X, X[selected[0]].reshape(1, -1)).flatten()

    for _ in range(1, min(k, n)):
        next_idx = np.argmax(min_dist)
        selected.append(next_idx)

        new_dist = cosine_distances(X, X[next_idx].reshape(1, -1)).flatten()
        min_dist = np.minimum(min_dist, new_dist)

    return selected

# =========================
# REPRESENTATIVE SELECTION PER CLUSTER
# =========================
def select_representatives(strata_to_idx, cluster_labels, embeddings, k_per_class=20):

    selected = []

    for c in np.unique(cluster_labels):

        if c == -1:
            continue  # noise cluster (HDBSCAN)

        strata_idx = np.where(cluster_labels == c)[0]

        # merge all refusals in these strata
        refusal_idx = np.concatenate([
            strata_to_idx[i] for i in strata_idx
        ])

        if len(refusal_idx) == 0:
            continue

        X = embeddings[refusal_idx]

        k = min(k_per_class, len(refusal_idx))

        chosen_local = k_center_greedy(X, k)

        selected.extend(refusal_idx[chosen_local])

    return selected

# =========================
# METRICS
# =========================
def compute_metrics(full_emb, selected_emb):

    full_emb = normalize(full_emb)
    selected_emb = normalize(selected_emb)

    sim = cosine_similarity(full_emb, selected_emb)
    coverage = np.mean(np.max(sim, axis=1))

    sim_sel = cosine_similarity(selected_emb)
    np.fill_diagonal(sim_sel, 0)

    diversity = 1 - np.mean(sim_sel)
    redundancy = np.mean(sim_sel)

    return coverage, diversity, redundancy

# =========================
# UMAP VISUALIZATION
# =========================
"""def plot_umap(embeddings, title, filename):

    reducer = umap.UMAP(metric="cosine", random_state=42)
    reduced = reducer.fit_transform(embeddings)

    plt.figure(figsize=(8, 6))
    plt.scatter(reduced[:, 0], reduced[:, 1])
    plt.title(title)
    plt.xlabel("UMAP-1")
    plt.ylabel("UMAP-2")
    plt.tight_layout()
    plt.savefig(filename, dpi=300)
    plt.close()"""

def plot_umap(embeddings, centroids, title, filename):

    combined = np.vstack([embeddings, centroids])

    reducer = umap.UMAP(metric="cosine", random_state=42)
    reduced = reducer.fit_transform(combined)

    n_emb = embeddings.shape[0]
    reduced_emb = reduced[:n_emb]
    reduced_centroids = reduced[n_emb:]

    # Assign each embedding to nearest centroid
    assignments = pairwise_distances_argmin(embeddings, centroids)

    # Compute density per centroid
    densities = np.array([
        np.sum(assignments == i) for i in range(len(centroids))
    ])

    # Scale circle sizes (matplotlib uses area, not radius)
    min_size, max_size = 100, 800
    if densities.max() == densities.min():
        sizes = np.full_like(densities, (min_size + max_size) / 2)
    else:
        sizes = min_size + (
            (densities - densities.min()) /
            (densities.max() - densities.min())
        ) * (max_size - min_size)

    plt.figure(figsize=(8, 6))

    # Embeddings
    plt.scatter(
        reduced_emb[:, 0],
        reduced_emb[:, 1],
        s=10,
        alpha=0.7,
        label="Embeddings"
    )

    # Centroids as circles with size ∝ density
    plt.scatter(
        reduced_centroids[:, 0],
        reduced_centroids[:, 1],
        s=sizes,
        c="red",
        alpha=0.30,
        edgecolors="black",
        linewidths=1,
        label="Centroids (size ∝ density)"
    )

    plt.title(title)
    plt.xlabel("UMAP-1")
    plt.ylabel("UMAP-2")
    plt.legend()
    plt.tight_layout()
    plt.savefig(filename, dpi=300)
    plt.close()


def get_best_config_strata(log_df, df, embeddings):

    best = log_df.iloc[0]

    best_method = best["method"]
    best_k = int(best["k"])
    best_centroids = best["cluster_centroids"]

    # -------------------------
    # USE ORIGINAL SELECTION
    # -------------------------
    best_idx = best["selected_idx"]

    # safety check
    if isinstance(best_idx, str):
        import ast
        best_idx = ast.literal_eval(best_idx)

    best_embeddings = embeddings[best_idx]

    print(f"🏆 Best config → {best_method}, k={best_k}")

    # -------------------------
    # BUILD FINAL DATAFRAME DIRECTLY
    # -------------------------
    final_df = df.iloc[best_idx].copy().reset_index(drop=True)

    best_centroids = normalize(best_centroids, axis=1)
    best_embeddings = normalize(best_embeddings, axis=1)

    return final_df, best_method, best_k, best_idx, best_centroids, best_embeddings

def run_ablation():
    
    print(f"Loading data from {INPUT_FILE}...")
    df = pd.read_csv(INPUT_FILE)

    # filter
    df = df[df["human_ensemble_judge"] == 0].copy()

    if USE_FULL_CONVERSATION:
        df["refusal"] = df.apply(get_full_conversation, axis=1)
    else:
        df["refusal"] = df.apply(get_final_response, axis=1)

    df = df[
        ["refusal", "subtopic", "tense", "turn_depth", "attack_name"]
    ].dropna().drop_duplicates().reset_index(drop=True)

    print(f" Candidates: {len(df)}")


    # --------------------------------------------------
    # isolate special case: turn_depth=3 & tense=present because of multitude of attack styles.
    # --------------------------------------------------
    mask = (
        (df["turn_depth"] == 3) &
        (df["tense"] == "present")
    )
    
    special_case = df[mask].copy()
    remaining = df[~mask].copy()
    
    # --------------------------------------------------
    # create joint strata: (subtopic, attack_name)
    # --------------------------------------------------
    special_case["stratum"] = (
        special_case["subtopic"] + "__" +
        special_case["attack_name"]
    )
    
    #print("\nBefore balancing:")
    #print(special_case["stratum"].value_counts())
    
    # smallest stratum count
    min_count = special_case["stratum"].value_counts().min()
    
    # balance special case
    balanced_special = (
        special_case
        .groupby("stratum", group_keys=False)
        .apply(lambda x: x.sample(min_count, random_state=42))
        .drop(columns="stratum")
        .reset_index(drop=True)
    )
    
    #print("\nAfter balancing:")
    """print(
        balanced_special
        .groupby(["subtopic", "attack_name"])
        .size()
    )"""
    
    # --------------------------------------------------
    # recombine and overwrite df
    # --------------------------------------------------
    df = (
        pd.concat([remaining, balanced_special], ignore_index=True)
        .sample(frac=1, random_state=42)
        .reset_index(drop=True)
    )
    
    print(f"\nFinal dataset size: {len(df)}")

    # =========================
    # EMBEDDINGS
    # =========================
    print(f"🔄 Encoding embeddings with {EMBEDDING_MODEL_NAME}...")
    set_seed(42)  # IMPORTANT for reproducibility
    embeddings = encode_texts(df["refusal"].tolist(), embedder, tokenizer)

    # -------------------------
    # STRATA LEVEL
    # -------------------------
    X_strata, strata_to_idx = build_strata_embeddings(df, embeddings)

    # -------------------------
    # ABLATION SPACE
    # -------------------------
    methods = ["kmeans", "hdbscan"]
    k_values = [5, 10, 20, 30, 40, 50]

    results = []

    for method in methods:
        for k in k_values:

            print(f"\nRunning: {method} | k={k}")

            try:
                # --------------------------
                # 1. clustering
                # --------------------------
                cluster_labels = cluster_strata(
                    X_strata,
                    method=method,
                    k=k
                )

                # =========================
                # NEW: cluster centroids
                # =========================
                cluster_centroids = compute_cluster_centroids(X_strata, cluster_labels)
                cluster_centroids = normalize(cluster_centroids, axis=1)
    
                n_clusters = len(set(cluster_labels)) - (1 if -1 in cluster_labels else 0)
                noise = np.sum(cluster_labels == -1)
    
                # --------------------------
                # 2. silhouette (SAFE)
                # --------------------------
                sil_score = None
    
                mask = cluster_labels != -1
                X_valid = X_strata[mask]
                y_valid = cluster_labels[mask]
    
                if len(set(y_valid)) > 1 and len(X_valid) > 1:
                    sil_score = silhouette_score(X_valid, y_valid)
                else:
                    sil_score = -1  # invalid clustering case
    
                # --------------------------
                # 3. selection
                # --------------------------
                selected_idx = select_representatives(
                    strata_to_idx,
                    cluster_labels,
                    embeddings,
                    k_per_class=20
                )
    
                if len(selected_idx) == 0:
                    print("Skipped: empty selection")
                    continue
    
                selected_emb = embeddings[selected_idx]
    
                coverage, diversity, redundancy = compute_metrics(
                    embeddings,
                    selected_emb
                )
    
                # --------------------------
                # 4. NORMALIZED FINAL SCORE
                # --------------------------
                # assume all metrics are in [0,1]
                score = (
                    0.4 * coverage +
                    0.4 * diversity -
                    0.2 * redundancy
                )
    
                results.append({
                    "method": method,
                    "k": k,
                    "coverage": coverage,
                    "diversity": diversity,
                    "redundancy": redundancy,
                    "n_clusters": n_clusters,
                    "noise": noise,
                    "silhouette_score": sil_score,
                    "score": score,
                    "selected_size": len(selected_idx),
                    "selected_idx": selected_idx,
                    "cluster_centroids": cluster_centroids
                })

                print("\n📊 Selection Quality Metrics:")
                print(f"Coverage Score   : {coverage:.4f} (higher is better)")
                print(f"Diversity Score  : {diversity:.4f} (higher is better)")
                print(f"Silhouette Score  : {sil_score:.4f} (higher is better)")
                print(f"Redundancy Score : {redundancy:.4f} (lower is better)")
                print(f"Score: {score:.4f}")

            except Exception as e:
                print(f"Failed: {method}, k={k}")
                print(e)

    # -------------------------
    # RESULTS
    # -------------------------
    log_df = pd.DataFrame(results)
    log_df = log_df.sort_values("score", ascending=False)

    log_file = OUTPUT_PREFIX + "ablation_results.csv"
    log_df.to_csv(log_file, index=False)

    print(f"\n Saved ablation log → {log_file}")

    best_df, best_method, best_k, best_idx, best_centroids, best_embeddings = get_best_config_strata(
    log_df,
    df,
    embeddings)

    # -------------------------
    # SAVE ONLY BEST DATASET
    # -------------------------
    output_file = OUTPUT_PREFIX + best_method +"_refusal_set.csv"
    best_df.to_csv(output_file, index=False)

    # =========================
    # OPTIONAL: FULL SNAPSHOT (REPRODUCIBILITY)
    # =========================
    joblib.dump(
        {
            "refusal_embeddings": best_embeddings,
            "selected_idx": best_idx,
            "cluster_centroids": best_centroids
        },
        OUTPUT_PREFIX + best_method + "_refusal_full_snapshot.pkl"
    )

    """plot_umap(
        best_embeddings,
        f"BEST: {best_method} (k={best_k})",
        OUTPUT_PREFIX + best_method +"_umap.png"
    )"""

    plot_umap(
        best_embeddings,
        best_centroids,
        f"BEST: {best_method} (k={best_k})",
        OUTPUT_PREFIX + best_method +"_umap.png"
    )

    print(f" Saved BEST → {output_file}")

def run_full_experiments(
    model_names,
    chunking_options=(False, True)
):
    global EMBEDDING_MODEL_NAME, USE_CHUNKING
    global embedder, tokenizer
    global RUN_SIGNATURE, EMB_MODEL_FOLDER, OUTPUT_PREFIX, CHUNK_TAG

    for model_name in model_names:
        for chunking in chunking_options:

            print("\n" + "="*60)
            print(f"🚀 Running: model={model_name} | chunking={chunking}")
            print("="*60)

            # -------------------------
            # UPDATE GLOBAL CONFIG
            # -------------------------
            EMBEDDING_MODEL_NAME = model_name
            USE_CHUNKING = chunking

            CHUNK_TAG = "chunked" if chunking else "no_chunk"

            RUN_SIGNATURE = (
                f"{EMBEDDING_MODEL_NAME}__{CHUNK_TAG}"
                f"__full-{USE_FULL_CONVERSATION}__{TRAINING_TEST}"
            )

            EMB_MODEL_FOLDER = os.path.join(TRAIN_DIR, EMBEDDING_MODEL_NAME)
            os.makedirs(EMB_MODEL_FOLDER, exist_ok=True)

            OUTPUT_PREFIX = os.path.join(
                EMB_MODEL_FOLDER,
                f"{RUN_SIGNATURE}__model_agnostic_"
            )

            # -------------------------
            # RELOAD EMBEDDING MODEL (CRITICAL)
            # -------------------------
            print(f"🔄 Loading embedder: {model_name}")
            embedder = SentenceTransformer(model_name)
            tokenizer = AutoTokenizer.from_pretrained(
                "sentence-transformers/" + model_name
            )

            # -------------------------
            # RUN ABLATION
            # -------------------------
            run_ablation()

            print(f"✅ Done: {model_name} | chunking={chunking}")

# =========================
# RUN
# =========================
#run_ablation()
run_full_experiments(
    model_names=EMB_MODEL_NAMES,
    chunking_options=[False, True]
)


🚀 Running: model=all-MiniLM-L6-v2 | chunking=False
🔄 Loading embedder: all-MiniLM-L6-v2
Loading data from training/Test_1_all_models.csv...
 Candidates: 1501

Final dataset size: 1007
🔄 Encoding embeddings with all-MiniLM-L6-v2...


/tmp/ipykernel_1935932/2566312725.py:551: FutureWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  .apply(lambda x: x.sample(min_count, random_state=42))


Batches:   0%|          | 0/16 [00:00<?, ?it/s]


Running: kmeans | k=5

📊 Selection Quality Metrics:
Coverage Score   : 0.6268 (higher is better)
Diversity Score  : 0.8534 (higher is better)
Silhouette Score  : 0.1900 (higher is better)
Redundancy Score : 0.1466 (lower is better)
Score: 0.5627

Running: kmeans | k=10

📊 Selection Quality Metrics:
Coverage Score   : 0.6556 (higher is better)
Diversity Score  : 0.8584 (higher is better)
Silhouette Score  : 0.2039 (higher is better)
Redundancy Score : 0.1416 (lower is better)
Score: 0.5773

Running: kmeans | k=20

📊 Selection Quality Metrics:
Coverage Score   : 0.7589 (higher is better)
Diversity Score  : 0.8182 (higher is better)
Silhouette Score  : 0.1307 (higher is better)
Redundancy Score : 0.1818 (lower is better)
Score: 0.5945

Running: kmeans | k=30

📊 Selection Quality Metrics:
Coverage Score   : 0.8144 (higher is better)
Diversity Score  : 0.8106 (higher is better)
Silhouette Score  : 0.0100 (higher is better)
Redundancy Score : 0.1894 (lower is better)
Score: 0.6121

Running:

/home/michael/venvs/torch-gpu/lib/python3.10/site-packages/umap/umap_.py:1952: UserWarning: n_jobs value 1 overridden to 1 by setting random_state. Use no seed for parallelism.
  warn(


...

Loading data from training/Test_1_all_models.csv...
 Candidates: 1501

Final dataset size: 1007
🔄 Encoding embeddings with all-roberta-large-v1...


Batches:   0%|          | 0/44 [00:00<?, ?it/s]


Running: kmeans | k=5

📊 Selection Quality Metrics:
Coverage Score   : 0.6056 (higher is better)
Diversity Score  : 0.8676 (higher is better)
Silhouette Score  : 0.1814 (higher is better)
Redundancy Score : 0.1324 (lower is better)
Score: 0.5628

Running: kmeans | k=10

📊 Selection Quality Metrics:
Coverage Score   : 0.6776 (higher is better)
Diversity Score  : 0.8559 (higher is better)
Silhouette Score  : 0.1822 (higher is better)
Redundancy Score : 0.1441 (lower is better)
Score: 0.5846

Running: kmeans | k=20

📊 Selection Quality Metrics:
Coverage Score   : 0.7509 (higher is better)
Diversity Score  : 0.8155 (higher is better)
Silhouette Score  : 0.1475 (higher is better)
Redundancy Score : 0.1845 (lower is better)
Score: 0.5897

Running: kmeans | k=30

📊 Selection Quality Metrics:
Coverage Score   : 0.8146 (higher is better)
Diversity Score  : 0.8122 (higher is better)
Silhouette Score  : 0.0305 (higher is better)
Redundancy Score : 0.1878 (lower is better)
Score: 0.6132

Running:

/home/michael/venvs/torch-gpu/lib/python3.10/site-packages/umap/umap_.py:1952: UserWarning: n_jobs value 1 overridden to 1 by setting random_state. Use no seed for parallelism.
  warn(


 Saved BEST → training/all-roberta-large-v1/all-roberta-large-v1__chunked__full-False__Test_1__model_agnostic_kmeans_refusal_set.csv
✅ Done: all-roberta-large-v1 | chunking=True
